# Scalable Student Grade Classification with PySpark

This notebook presents the cleaned portfolio version of the CSC 555 project. It focuses on the distributed data-processing workflow, multiclass model benchmarking, class imbalance, and an important methodological finding: **`total_score` is closely tied to the definition of `grade`, so the original feature set contains target leakage.**

> The reported course metrics are preserved as historical results. The notebook does not claim that those metrics represent a leakage-free production model.

## 1. Setup and data loading

In [ ]:
import os
from pyspark.sql import SparkSession

from src.config import DATA_PATH, RANDOM_SEED
from src.data_preprocessing import load_data, clean_data, null_counts

spark = SparkSession.builder.appName("StudentGradeClassification").getOrCreate()
print(f"Using data: {DATA_PATH}")

df_raw = load_data(spark, DATA_PATH)
print("Raw rows:", df_raw.count())
df_raw.printSchema()

## 2. Data quality and preprocessing

In [ ]:
display(null_counts(df_raw))
df = clean_data(df_raw)
print("Cleaned rows:", df.count())
df.show(5, truncate=False)

The original analysis found 10,000 raw records, 213 rows containing at least one null, and no duplicate rows after cleaning, leaving **9,787 records**.

## 3. Exploratory analysis

In [ ]:
numeric_cols_eda = ["math_score", "reading_score", "writing_score", "science_score", "total_score"]
df.describe(numeric_cols_eda).show()

grade_counts = df.groupBy("grade").count().orderBy("grade")
grade_counts.show()

In [ ]:
import matplotlib.pyplot as plt

label_pd = grade_counts.toPandas()
label_pd.plot(kind="bar", x="grade", y="count", legend=False, figsize=(8, 5))
plt.title("Grade Distribution")
plt.xlabel("Grade")
plt.ylabel("Students")
plt.tight_layout()
plt.show()

In [ ]:
score_pd = df.select("math_score", "grade").toPandas()
plt.figure(figsize=(8, 5))
score_pd.boxplot(column="math_score", by="grade")
plt.suptitle("")
plt.title("Math Score Distribution by Grade")
plt.xlabel("Grade")
plt.ylabel("Math Score")
plt.tight_layout()
plt.show()

### EDA takeaway

The target is highly imbalanced: the original analysis found 5,544 `B` records but only 61 `Fail` records. Academic scores also show a strong relationship with grade. Test-preparation completion showed only a small difference in average score in the original analysis.

## 4. Feature engineering

In [ ]:
from src.feature_engineering import add_grade_index, add_class_weights, encode_features

df = add_grade_index(df)
df = add_class_weights(df)
df_ml, numeric_cols, categorical_cols, assembler = encode_features(df)

print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)
df_ml.select("features", "label", "classWeightCol").show(3, truncate=False)

### Class imbalance

Class weights use inverse class frequency, matching the original Random Forest workflow. The rare `Fail` class therefore receives substantially more weight than the dominant `B` class.

## 5. Train/test split

In [ ]:
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=RANDOM_SEED)
print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

## 6. Random Forest baseline

In [ ]:
from src.modeling import train_random_forest
from src.evaluation import evaluate, confusion_matrix

rf_model = train_random_forest(train_df)
rf_predictions = rf_model.transform(test_df)
rf_accuracy, rf_f1 = evaluate(rf_predictions)
print(f"Random Forest accuracy: {rf_accuracy:.4f}")
print(f"Random Forest F1:       {rf_f1:.4f}")
print("Confusion matrix:
", confusion_matrix(rf_predictions))

## 7. Model comparison

In [ ]:
from src.modeling import compare_models

results, fitted_models = compare_models(train_df, test_df)
results_df = spark.createDataFrame(results, ["Model", "Accuracy", "F1 Score"])
results_df.orderBy("Accuracy", ascending=False).show()

### Original course results

The submitted analysis reported approximately **90.96% / 89.55%** for Random Forest, **99.84% / 99.84%** for Logistic Regression, and **97.24% / 97.14%** for Decision Tree (accuracy / F1).

## 8. Feature importance

In [ ]:
importance = rf_model.featureImportances.toArray()
feature_names = assembler.getInputCols()

# The original notebook could only map importance values reliably to the assembler's
# input columns when dimensionality matched. Keep the analysis conservative here.
print("Number of model dimensions:", len(importance))
print("Assembler inputs:", feature_names)

## 9. Hyperparameter tuning

In [ ]:
from src.modeling import tune_random_forest

cv_model = tune_random_forest(train_df)
best_rf = cv_model.bestModel
tuned_predictions = best_rf.transform(test_df)
tuned_accuracy, tuned_f1 = evaluate(tuned_predictions)
print(f"Tuned Random Forest accuracy: {tuned_accuracy:.4f}")
print(f"Tuned Random Forest F1:       {tuned_f1:.4f}")

## 10. Methodological finding: target leakage

In [ ]:
print("Features include total_score:", "total_score" in numeric_cols)
print("Target column:", "grade")

`total_score` is an aggregate of the subject scores, while the project report describes grade thresholds based on total score. The original Random Forest analysis also identified `total_score` as the dominant feature (~0.50 importance). Therefore, the very high Logistic Regression result should **not** be presented as evidence of a robust early-warning system.

### What a stronger next experiment would do

- Remove `total_score` from the feature set.
- Define clearly which information is available at prediction time.
- Rebuild preprocessing and model evaluation from that feature set.
- Report per-class precision, recall, and F1, especially for `Fail`.

That leakage-free experiment is intentionally not fabricated here because the original dataset was not included with the source files used to prepare this repository.

## 11. Proposed production architecture

The original report proposed a **theoretical** AWS architecture:

`S3 → Airflow → EMR/Spark → S3 → Athena → QuickSight`

This is an architectural proposal, not a deployment completed in the course project.

## 12. Conclusion

The project demonstrates a practical Spark ML workflow from raw-data quality checks through feature engineering, model benchmarking, evaluation, and a scalable deployment concept. The most valuable technical lesson is that **high model accuracy is not sufficient**: feature provenance and target definition must be checked to avoid leakage.

The next iteration should prioritize a leakage-free prediction setup and class-level evaluation before considering production deployment.